# Teams history generator

Load every match JSON file, validate its structure and columns, merge all seasons, and store the result in `data/csv/teams_history.csv`.

In [75]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

JSON_DIR = Path("../data/json")
OUTPUT_PATH = Path("../data/csv/teams_history.csv")

## Discover and validate JSON files

In [76]:
json_files = sorted(JSON_DIR.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {JSON_DIR.resolve()}")

pd.DataFrame({"json_file": [path.name for path in json_files]})

,json_file
0,matches23-24.json
1,matches24-25.json
2,matches25-26.json


In [77]:
payloads = {}
validation_errors = []

for json_path in json_files:
    try:
        with json_path.open(encoding="utf-8") as json_file:
            payload = json.load(json_file)

        if not isinstance(payload, dict):
            raise TypeError("The root element must be an object")
        if not isinstance(payload.get("matches"), list):
            raise TypeError("The 'matches' field must be a list")

        payloads[json_path.name] = payload
    except (json.JSONDecodeError, OSError, TypeError) as error:
        validation_errors.append({"json_file": json_path.name, "error": str(error)})

if validation_errors:
    display(pd.DataFrame(validation_errors))
    raise ValueError("Invalid JSON files found. Fix them before generating teams_history.csv.")

print(f"Validated {len(payloads)} JSON files.")

Validated 3 JSON files.


## Convert every JSON file to a DataFrame

In [78]:
dataframes: dict[str, pd.DataFrame] = {}

for json_file_name, payload in payloads.items():
    matches_df = pd.json_normalize(payload["matches"], sep="_")
    matches_df.insert(0, "competition", payload.get("name", ""))
    matches_df.insert(1, "source_file", json_file_name)
    dataframes[json_file_name] = matches_df

pd.DataFrame(
    {
        "json_file": file_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
    }
    for file_name, dataframe in dataframes.items()
)

,json_file,rows,columns
0,matches23-24.json,380,9
1,matches24-25.json,380,9
2,matches25-26.json,380,10


## Check column correspondence

In [79]:
for file_name, df in dataframes.items():
    print(f"{file_name} cols: {df.columns}")

matches23-24.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score', 'score_ft', 'score_ht'],
      dtype='str')


In [80]:
df = dataframes["matches25-26.json"].copy()
df = df[df["score"].isna()]
(df[df["score"].isna()].shape[0], df.shape[0])

(344, 344)

In [ ]:
dataframes["matches25-26.json"].drop(columns=["score", "source_file"], inplace=True)

In [82]:
for file_name, df in dataframes.items():
    print(f"{file_name} cols: {df.columns}")

matches23-24.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ft', 'score_ht'],
      dtype='str')


In [83]:
for file_name, df in dataframes.items():
    dataframes[file_name] = df.sort_index(axis=1, ascending=True)
    print(f"{file_name} cols: {dataframes[file_name].columns}")

matches23-24.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'source_file',
       'team1', 'team2', 'time'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'source_file',
       'team1', 'team2', 'time'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'source_file',
       'team1', 'team2', 'time'],
      dtype='str')


In [84]:
reference_file = next(iter(dataframes))
reference_columns = dataframes[reference_file].columns.tolist()
reference_column_set = set(reference_columns)

column_checks = []
for file_name, dataframe in dataframes.items():
    current_columns = dataframe.columns.tolist()
    current_column_set = set(current_columns)
    column_checks.append(
        {
            "json_file": file_name,
            "same_columns": current_column_set == reference_column_set,
            "same_order": current_columns == reference_columns,
            "missing_columns": sorted(reference_column_set - current_column_set),
            "extra_columns": sorted(current_column_set - reference_column_set),
        }
    )

column_check_df = pd.DataFrame(column_checks)
display(column_check_df)

,json_file,same_columns,same_order,missing_columns,extra_columns
0,matches23-24.json,True,True,[],[]
1,matches24-25.json,True,True,[],[]
2,matches25-26.json,True,True,[],[]


## Merge and inspect the complete history

In [85]:
if not column_check_df[["same_columns", "same_order"]].all(axis=None):
    raise ValueError("The JSON DataFrames do not have matching ordered columns.")

teams_history = pd.concat(dataframes.values(), ignore_index=True)
teams_history = teams_history.sort_values(["date", "time", "team1", "team2"]).reset_index(drop=True)

print(f"Rows: {len(teams_history):,}")
print(f"Columns: {len(teams_history.columns)}")
display(teams_history.head())
display(teams_history.tail())

Rows: 1,140
Columns: 9


,competition,date,round,score_ft,score_ht,source_file,team1,team2,time
0,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[0, 1]","[0, 0]",matches23-24.json,Empoli FC,Hellas Verona FC,18:30
1,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[1, 3]","[1, 2]",matches23-24.json,Frosinone Calcio,SSC Napoli,18:30
2,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[2, 0]","[1, 0]",matches23-24.json,FC Internazionale Milano,AC Monza,20:45
3,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[1, 4]","[0, 3]",matches23-24.json,Genoa CFC,ACF Fiorentina,20:45
4,Italian Serie A 2023/24,2023-08-20,Matchday 1,"[2, 2]","[1, 1]",matches23-24.json,AS Roma,US Salernitana 1919,18:30


,competition,date,round,score_ft,score_ht,source_file,team1,team2,time
1135,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 2]","[1, 1]",matches25-26.json,AC Milan,Cagliari Calcio,20:45
1136,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[0, 2]","[0, 0]",matches25-26.json,Hellas Verona FC,AS Roma,20:45
1137,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 4]","[0, 1]",matches25-26.json,US Cremonese,Como 1907,20:45
1138,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 0]","[1, 0]",matches25-26.json,US Lecce,Genoa CFC,20:45
1139,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[2, 2]","[0, 1]",matches25-26.json,Torino FC,Juventus FC,21:45


# Build the history wins, loss, goals done and goals against DataFrame

In [ ]:
teams_history: pd.DataFrame = pd.DataFrame({
    "team": None,
    "season": None,
    "wins": None,
    "loss": None,
    "goals": None,
    "goals_per90": None,
    "goals_against": None,
    "goals_against_per90": None,
    "goals_fh": None,
    "goals_sh": None,
    "goals_against_fh": None,
    "goals_against_sh": None,
    "ts_goals": None,
    "ts_goals_against": None
})

## Store `teams_history.csv`

In [86]:
#OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#teams_history.to_csv(OUTPUT_PATH, index=False)

#print(f"Stored {len(teams_history):,} rows in {OUTPUT_PATH.resolve()}")